In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from segmentation.downloader import LidcIdriDownloader
from segmentation.patient_manager import PatientManager

## Get patients from BDD LIBDC-IDRI

In [ ]:
N_PATIENTS = 100
patient_ids = [f'LIDC-IDRI-{i:04d}' for i in range(1, N_PATIENTS + 1)]

dl = LidcIdriDownloader("./LIDC_data/", patient_ids)
dl.fill_patients_files_info()

In [ ]:
set(dl.patients_files_info.keys()) == set(patient_ids)

In [ ]:
dl.download()

## Work on patients

In [ ]:
dl.get_patient_files_info(patient_ids[3])

In [ ]:
patient0 = PatientManager(patient_ids[3], dl)

In [ ]:
patient0.init()

In [ ]:
patient0.display_volume_with_annotations()

## Segmentation

In [ ]:
from segmentation.segmenter import Segmenter
seg = Segmenter(patient0)
candidates = seg.run()

In [ ]:
slice_idx = 76
seg.display_lung_mask(slice_idx)
seg.display_segmentation(slice_idx)

In [ ]:
seg.display_candidates_3d()

### Tests

In [ ]:
ann_candidates = patient0.get_annotation_candidates()

In [ ]:
annotations_mask = patient0.get_volume_with_annotations()
ann_candidates = patient0.get_annotation_candidates()

offset = seg.get_roi_offset()
pairs = Segmenter.match_candidates(ann_candidates, seg.candidates, roi_offset=offset)
iou_scores = [
    Segmenter.compute_iou_3d(ann, cand, annotations_mask, seg.nodules_mask, offset)
    for ann, cand in pairs
]

In [ ]:
pairs

In [ ]:
iou_scores

In [ ]:
recall = len(pairs) / len(ann_candidates)
print(f"Recall: {recall:.2%} - {len(pairs)}/{len(ann_candidates)} annotations matched")